<a href="https://colab.research.google.com/github/mkweisbr/Portfolio/blob/main/clean_population_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import files

uploaded = files.upload()  # This will open a file picker

Saving P_Data_Extract_From_Population_estimates_and_projections.csv to P_Data_Extract_From_Population_estimates_and_projections (1).csv


In [5]:
# Basic imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("P_Data_Extract_From_Population_estimates_and_projections.csv")  # Use the exact filename
df.head()

,Country Name,Country Code,Series Name,Series Code,2025 [YR2025]
0,Afghanistan,AFG,Urban population,SP.URB.TOTL,11373235
1,Afghanistan,AFG,Rural population,SP.RUR.TOTL,32470876
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,43844111
3,Afghanistan,AFG,Number of infant deaths,SH.DTH.IMRT,..
4,Afghanistan,AFG,Net migration,SM.POP.NETM,-74404


In [6]:
# Reshaped messy, stacked data into a structured wide format. This enabled efficient analysis, aggregation, and visualization in Python using pandas.

df_wide = df.pivot_table(index='Country Name', columns='Series Name', values='2025 [YR2025]',aggfunc = 'first').reset_index()
df_wide.head()

Series Name,Country Name,Age dependency ratio (% of working-age population),"Birth rate, crude (per 1,000 people)","Death rate, crude (per 1,000 people)","Fertility rate, total (births per woman)",Net migration,Number of infant deaths,"Population, total",Rural population,Urban population
0,Afghanistan,81.77357224,34.391,5.608,4.659,-74404,..,43844111,32470876,11373235
1,Albania,52.04106334,9.898,8.634,1.33,-24230,..,2355297,968699,1386598
2,Algeria,58.05636499,18.034,4.645,2.67,-27531,..,47435312,11470458,35964854
3,American Samoa,53.57845918,14.737,7.5,2.246,-1051,..,46029,8970,37059
4,Andorra,39.23850792,6.854,6.223,1.096,843,..,82904,9161,73743


In [7]:
# Understanding the data type

df_wide.dtypes

,0
Series Name,
Country Name,object
Age dependency ratio (% of working-age population),object
"Birth rate, crude (per 1,000 people)",object
"Death rate, crude (per 1,000 people)",object
"Fertility rate, total (births per woman)",object
Net migration,object
Number of infant deaths,object
"Population, total",object
Rural population,object


In [8]:
# Replacing .. with NA

df_wide.replace('..', pd.NA, inplace=True)


In [9]:
# List of columns that should be numeric
numeric_cols = ['Urban population', 'Rural population', 'Population, total',
                'Net migration', 'Death rate, crude (per 1,000 people)', 'Birth rate, crude (per 1,000 people)',
                'Fertility rate, total (births per woman)', 'Age dependency ratio (% of working-age population)']

# Convert them to numbers
df_wide[numeric_cols] = df_wide[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [10]:
# Count missing values per column
df_wide.isnull().sum()

,0
Series Name,
Country Name,0
Age dependency ratio (% of working-age population),1
"Birth rate, crude (per 1,000 people)",0
"Death rate, crude (per 1,000 people)",0
"Fertility rate, total (births per woman)",0
Net migration,0
Number of infant deaths,217
"Population, total",1
Rural population,1


In [11]:

# Quick look at the data
df_wide.describe()

Series Name,Age dependency ratio (% of working-age population),"Birth rate, crude (per 1,000 people)","Death rate, crude (per 1,000 people)","Fertility rate, total (births per woman)",Net migration,"Population, total",Rural population,Urban population
count,216.000000,217.000000,217.000000,217.000000,2.170000e+02,2.160000e+02,2.160000e+02,2.160000e+02
mean,57.405300,17.088304,7.854249,2.335088,1.127143e+02,3.790400e+07,1.601518e+07,2.188882e+07
std,14.429281,9.293017,2.839517,1.144139,1.915214e+05,1.437240e+08,7.361215e+07,7.822340e+07
min,20.049624,4.759000,0.964000,0.686000,-1.235336e+06,9.492000e+03,0.000000e+00,5.956000e+03
25%,48.324379,9.622000,6.036000,1.478000,-1.469300e+04,8.717965e+05,2.096590e+05,4.692678e+05
50%,55.145478,13.343000,7.392000,1.938000,-1.051000e+03,6.772014e+06,1.636312e+06,4.251932e+06
75%,65.700770,23.876000,9.534000,3.027000,2.859000e+03,2.756419e+07,9.737032e+06,1.570588e+07
max,103.585627,45.361000,19.418000,5.938000,1.702358e+06,1.463866e+09,9.414243e+08,9.326299e+08


In [12]:
#renaming columns

df_wide = df_wide.rename(columns={
    'Country Name': 'country',
    'Age dependency ratio (% of working-age population)': 'dependency_ratio',
    'Birth rate, crude (per 1,000 people)': 'birth_rate',
    'Death rate, crude (per 1,000 people)': 'death_rate',
    'Fertility rate, total (births per woman)': 'fertility_rate',
    'Net migration': 'net_migration',
    'Number of infant deaths': 'infant_deaths',
    'Population, total': 'population_total',
    'Rural population': 'rural_population',
    'Urban population': 'urban_population'
})

In [18]:
# Normalize columns to help calculate a risk score
# Fertility normalization
df_wide['fertility_norm'] = (
    (df_wide['fertility_rate'] - df_wide['fertility_rate'].min()) /
    (df_wide['fertility_rate'].max() - df_wide['fertility_rate'].min())
)

# Dependency ratio normalization
df_wide['dependency_norm'] = (
    (df_wide['dependency_ratio'] - df_wide['dependency_ratio'].min()) /
    (df_wide['dependency_ratio'].max() - df_wide['dependency_ratio'].min())
)

# Migration normalization
df_wide['migration_norm'] = (
    (df_wide['net_migration'] - df_wide['net_migration'].min()) /
    (df_wide['net_migration'].max() - df_wide['net_migration'].min())
)

# Create risk score
df_wide['population_risk_score'] = (
    df_wide['dependency_norm'] +
    (1 - df_wide['fertility_norm']) +
    (1 - df_wide['migration_norm'])
)
